# 6.4. SS.com pilna rasmošanas plūsma ar funkcijām

[![Atvērt Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_BDAA_Course_2026/blob/main/notebooks/lecture_06_web_scraping/06_04_ss_com_full_scraping_workflow.ipynb)

Šī darba burtnīca noslēdz tīmekļa rasmošanas tēmu un sagatavo datus nākamajai lekcijai par **datu analīzi un vizualizāciju**.

Galvenā ideja ir pārveidot iepriekšējo eksperimentālo rasmošanas kodu par skaidru, atkārtoti izmantojamu funkciju plūsmu:

**URL → HTTP pieprasījums → HTML → BeautifulSoup → sludinājumu rindas → DataFrame → CSV/XLSX**

Darba burtnīca ir veidota kā **Run All** piemērs. Parastā lietošanā jāmaina tikai konstante `START_URL`. Piemēram:

```python
START_URL = "https://www.ss.com/en/real-estate/flats/riga/centre/sell/"
```

Pēc tam pēdējā šūna automātiski:

1. ielādē pirmo rezultātu lapu;
2. no tās pašas HTML atbildes nosaka, cik lapu ir kategorijā;
3. apstrādā pirmās lapas sludinājumus, neveicot otru HTTP pieprasījumu;
4. pa vienai ielādē atlikušās lapas;
5. apvieno visus sludinājumus vienā `DataFrame`;
6. saglabā rezultātu gan CSV, gan XLSX formātā.

Svarīgs dizaina princips: **katrs rezultātu lapas URL tiek pieprasīts tikai vienu reizi**.

## Kāpēc HTTP un HTML apstrādi atdalām?

Rasmošanas programmā ir divi atšķirīgi darba veidi:

- **I/O jeb ievades/izvades darbības** — HTTP pieprasījumi internetā un failu saglabāšana diskā;
- **datu apstrāde** — jau saņemta HTML analizēšana, kolonnu atrašana, sludinājumu rindu pārveidošana un `DataFrame` veidošana.

Šajā notebook tikai `fetch_soup()` veic HTTP pieprasījumu. Savukārt `process_page()`, `process_headline()`, `find_ad_rows()` un citas HTML funkcijas strādā ar jau saņemtu `BeautifulSoup` objektu.

Tas ļauj pirmās lapas HTML izmantot **divreiz lokāli** — gan lapu skaita noteikšanai, gan sludinājumu iegūšanai — bet pašu lapu no servera pieprasīt tikai vienu reizi.

## Vide un bibliotēkas

Notebook paredzēts darbam gan lokāli VS Code/Jupyter vidē, gan Google Colab.

Nepieciešamās ārējās bibliotēkas:

- `requests` — HTTP pieprasījumiem;
- `beautifulsoup4` — HTML parsēšanai;
- `pandas` — tabulāru datu veidošanai un eksportam;
- `lxml` — ātram HTML parserim;
- `openpyxl` — XLSX failu saglabāšanai.

Nākamā šūna instalē tikai tās pakotnes, kuru konkrētajā vidē trūkst.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "lxml": "lxml",
    "openpyxl": "openpyxl",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Instalējam trūkstošās pakotnes:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Visas nepieciešamās pakotnes jau ir instalētas.")

print("Python:", sys.version.split()[0])
print(
    "Vide:",
    "Google Colab" if "google.colab" in sys.modules else "lokāls Jupyter/VS Code",
)

## Importi un konfigurācija

Šajā šūnā atrodas visas galvenās konfigurācijas konstantes. Ikdienas lietošanā studentam būtu jāmaina tikai `START_URL`.

`REQUEST_DELAY` nosaka pauzi starp secīgiem HTTP pieprasījumiem. Tā ir svarīga pieklājīgas rasmošanas prakse: nav nepieciešams serverim nosūtīt desmitiem pieprasījumu pēc iespējas ātrāk.

`OUTPUT_DIR` nosaka mapi, kurā tiks saglabāti rezultāti.

In [ ]:
from datetime import datetime
from pathlib import Path
import re
from time import sleep
from urllib.parse import unquote, urljoin, urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; RTU-BDAA-teaching-example/1.0)"
}

REQUEST_TIMEOUT = 20
REQUEST_DELAY = 0.5
OUTPUT_DIR = Path("data")

# Parastā lietošanā mainām tikai šo URL.
START_URL = "https://www.ss.com/en/real-estate/flats/riga/centre/sell/"

print("Rasmošanas sākuma URL:")
print(START_URL)

## 1. Funkcija `fetch_soup(url)`

`fetch_soup()` ir vienīgā funkcija šajā rasmošanas plūsmā, kuras uzdevums ir **saņemt HTML no interneta**.

Funkcija:

1. saņem vienu URL kā argumentu;
2. izpilda `requests.get()` ar iepriekš definētu `User-Agent` un timeout;
3. ar `raise_for_status()` pārtrauc darbu, ja serveris atgriež HTTP kļūdu, piemēram, `404`, `403` vai `429`;
4. saņemto HTML tekstu pārvērš par `BeautifulSoup` objektu;
5. atgriež tikai šo vienu objektu.

Tādējādi visas pārējās HTML apstrādes funkcijas nav atkarīgas no interneta. Tās var apstrādāt jebkuru jau iegūtu `BeautifulSoup` objektu.

Ja vietne atgriež `429 Too Many Requests`, nevajag automātiski palielināt pieprasījumu skaitu vai nekavējoties mēģināt vēlreiz. Jāsamazina pieprasījumu biežums un jāievēro vietnes piekļuves noteikumi.

In [ ]:
def fetch_soup(url: str) -> BeautifulSoup:
    """Lejupielādē vienu HTML lapu un atgriež BeautifulSoup objektu."""
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")

## 2. Funkcija `process_headline(soup)`

SS.com sludinājumu tabulas kolonnu nosaukumi atrodas galvenes rindā ar `id="head_line"`.

Šīs funkcijas uzdevums ir **nolasīt lapas shēmu**, nevis pieņemt, ka vienmēr rasmojam tieši dzīvokļus. Dažādās SS.com kategorijās var būt atšķirīgas kolonnas, piemēram, dzīvokļiem ir platība un stāvs, bet automašīnām — gads, motors vai nobraukums.

Papildus lapā redzamajām kolonnām mēs vienmēr pievienojam divus savus laukus:

- `description` — sludinājuma īsais apraksts no rezultātu tabulas;
- `url` — pilnā saite uz konkrēto sludinājumu.

Ja galvenes rinda nav atrasta, funkcija izmet `ValueError`. Tas ir labāk nekā klusi izveidot nepareizu tabulu: HTML struktūras maiņa ir normāla web scraping dzīves cikla daļa, un šādā gadījumā kods ir jāpārbauda.

In [ ]:
def process_headline(soup: BeautifulSoup) -> list[str]:
    """No SS.com tabulas galvenes iegūst rezultātu kolonnu nosaukumus."""
    headline = soup.find("tr", id="head_line")

    if headline is None:
        raise ValueError("Neizdevās atrast tabulas galveni ar id='head_line'.")

    header_cells = headline.find_all("td")
    page_columns = [
        cell.get_text(" ", strip=True)
        for cell in header_cells[1:]
    ]

    return ["description", "url", *page_columns]

## 3. Funkcija `find_ad_rows(soup)`

Vienā SS.com lapā ir daudz `<tr>` elementu, taču ne visi ir sludinājumi. Rezultātu tabulas īstajiem sludinājumiem ir `id`, kas sākas ar `tr_`.

Tajā pašā laikā var būt arī reklāmas jeb banneru rindas, piemēram, ar `id`, kas sākas ar `tr_bnr`. Tās mūsu datu kopā nav vajadzīgas.

`find_ad_rows()` veic tikai **HTML elementu atlasi**:

- atrod visas `<tr>` rindas;
- patur tās, kuru `id` sākas ar `tr_`;
- izslēdz banneru rindas;
- atgriež BeautifulSoup `Tag` objektu sarakstu.

Šajā funkcijā vēl netiek mēģināts interpretēt atsevišķo šūnu saturu. Atlase un datu interpretācija apzināti ir divi dažādi soļi.

In [ ]:
def find_ad_rows(soup: BeautifulSoup) -> list:
    """Atrod SS.com rezultātu tabulas rindas, kas reprezentē sludinājumus."""
    rows = soup.find_all("tr")

    return [
        row
        for row in rows
        if row.get("id", "").startswith("tr_")
        and not row.get("id", "").startswith("tr_bnr")
    ]

## 4. Funkcija `process_ad_row(row, columns)`

Šī ir galvenā zemākā līmeņa datu pārveidošanas funkcija: **viena HTML sludinājuma rinda kļūst par vienu Python vārdnīcu**.

SS.com rezultātu rindā viena no pirmajām šūnām satur saiti uz pilno sludinājumu, nākamā satur īso aprakstu, bet atlikušās šūnas atbilst kolonnām, kuras ieguvām ar `process_headline()`.

Relatīvā saite, piemēram, `/msg/en/...`, ar `urljoin()` tiek pārvērsta pilnā URL.

Funkcija veic arī nelielu aizsardzības pārbaudi. Ja rindā nav pietiekami daudz `<td>` šūnu vai nav atrodamas saites, tiek atgriezta tukša vārdnīca. Augstāka līmeņa funkcija šādu rezultātu vienkārši neiekļaus datu kopā.

Šeit mēs vēl **netīrām analītiskās vērtības**. Piemēram, `"235,000 €"` paliek teksts. Šādu kolonnu tipizēšana būs nākamās lekcijas datu tīrīšanas uzdevums.

In [ ]:
def process_ad_row(row, columns: list[str]) -> dict:
    """Pārvērš vienu SS.com sludinājuma <tr> elementu par vārdnīcu."""
    cells = row.find_all("td")

    if len(cells) < 3:
        return {}

    link = row.find("a", href=True)
    if link is None:
        return {}

    ad = {
        "description": cells[2].get_text(" ", strip=True),
        "url": urljoin("https://www.ss.com", link["href"]),
    }

    for cell, column in zip(cells[3:], columns[2:]):
        ad[column] = cell.get_text(" ", strip=True)

    return ad

## 5. Funkcija `process_all_ads(soup, columns)`

Kad protam apstrādāt vienu sludinājuma rindu, nākamais solis ir to pašu darbību piemērot **visām sludinājumu rindām vienā lapā**.

`process_all_ads()`:

1. ar `find_ad_rows()` iegūst visas atbilstošās HTML rindas;
2. katrai rindai izsauc `process_ad_row()`;
3. ignorē tukšas vārdnīcas, kas varētu rasties no neparedzētas vai nepilnīgas rindas;
4. atgriež sarakstu ar vārdnīcām.

Rezultāts ir ļoti ērts Pandas bibliotēkai: `list[dict]` var tieši pārvērst par `DataFrame`.

Šeit redzams bieži sastopams programmēšanas princips: **vispirms uzrakstām funkciju vienam objektam, pēc tam šo funkciju pielietojam daudziem objektiem**.

In [ ]:
def process_all_ads(soup: BeautifulSoup, columns: list[str]) -> list[dict]:
    """Apstrādā visas vienas SS.com rezultātu lapas sludinājumu rindas."""
    ad_rows = find_ad_rows(soup)
    ads = []

    for row in ad_rows:
        ad = process_ad_row(row, columns)
        if ad:
            ads.append(ad)

    return ads

## 6. Funkcija `process_page(soup)`

`process_page()` apvieno iepriekšējās HTML funkcijas un izveido **vienas jau lejupielādētas lapas `DataFrame`**.

Svarīgi: funkcija saņem `BeautifulSoup`, nevis URL. Tātad tā **neveic HTTP pieprasījumu**.

Darbības ir šādas:

1. `process_headline()` nosaka kolonnu nosaukumus;
2. `process_all_ads()` iegūst visu sludinājumu vārdnīcas;
3. `pd.DataFrame()` izveido tabulu ar paredzēto kolonnu secību.

Šis nodalījums ir būtisks mūsu vienreizējā HTTP pieprasījuma principam. Pirmās lapas `soup` varam nodot gan `get_last_page_number()`, gan `process_page()` — abas funkcijas strādā ar to pašu lokālo HTML objektu.

In [ ]:
def process_page(soup: BeautifulSoup) -> pd.DataFrame:
    """Pārvērš vienas jau lejupielādētas SS.com lapas HTML par DataFrame."""
    columns = process_headline(soup)
    ads = process_all_ads(soup, columns)
    return pd.DataFrame(ads, columns=columns)

## 7. Funkcija `get_last_page_number(soup)`

Lai savāktu visu kategoriju, jāzina, cik rezultātu lapu ir pieejamas.

SS.com lapošanas saitēs lapas numurs ir URL daļa, piemēram `page2.html`, `page9.html` vai `page29.html`. Tomēr pirmajā lapā redzamais numurēto saišu bloks parasti rāda tikai pirmās dažas lapas, tāpēc vienkārši paņemt lielāko redzamo numuru nebūtu pietiekami.

SS.com pirmajā rezultātu lapā saite ar `rel="prev"` (interfeisā **Iepriekšējie / Previous**) ved uz pēdējo rezultātu lapu. Tāpēc funkcija:

1. vispirms meklē `a` elementu ar `rel="prev"` un no tā URL iegūst pēdējās lapas numuru;
2. papildus savāc arī parasto `pageN.html` saišu numurus kā rezerves variantu;
3. atgriež lielāko atrasto lapas numuru;
4. ja lapošanas saišu nav, atgriež `1`.

Šī ir SS.com specifiska lapošanas īpatnība, tāpēc tā ir izolēta vienā funkcijā. Ja vietne nākotnē mainīs lapošanas HTML, būs jālabo tikai šī funkcija.

Būtiski: funkcija saņem jau esošu `soup`. Tā **nepieprasa pirmo lapu vēlreiz**.

In [ ]:
def get_last_page_number(soup: BeautifulSoup) -> int:
    """No SS.com lapošanas saitēm nosaka pēdējās rezultātu lapas numuru."""
    page_numbers = []

    # SS.com pirmajā lapā rel="prev" saite ved uz pēdējo lapu.
    previous_link = soup.find("a", rel="prev")
    if previous_link and previous_link.get("href"):
        match = re.search(
            r"(?:^|/)page(\d+)\.html(?:$|[?#])",
            previous_link["href"],
        )
        if match:
            page_numbers.append(int(match.group(1)))

    # Rezerves variants: pārbaudām arī visas pārējās lapošanas saites.
    for anchor in soup.find_all("a", href=True):
        match = re.search(
            r"(?:^|/)page(\d+)\.html(?:$|[?#])",
            anchor["href"],
        )
        if match:
            page_numbers.append(int(match.group(1)))

    return max(page_numbers, default=1)

## 8. Funkcija `get_all_page_urls(start_url, first_soup)`

Šī funkcija izveido pilnu rezultātu lapu URL sarakstu.

Tai ir divi argumenti:

- `start_url` — kategorijas pirmās lapas adrese;
- `first_soup` — **jau iepriekš lejupielādētās** pirmās lapas HTML.

No `first_soup` ar `get_last_page_number()` iegūstam pēdējās lapas numuru. Pēc tam URL adreses `page2.html`, `page3.html` utt. tiek tikai **konstruētas kā teksta virknes** — to izveide vēl neveic nevienu HTTP pieprasījumu.

Funkcija saglabā arī sākuma URL query parametrus, ja tādi ir. Ja kategorijai ir tikai viena lapa, rezultāts ir saraksts ar vienu elementu — `start_url`.

In [ ]:
def get_all_page_urls(start_url: str, first_soup: BeautifulSoup) -> list[str]:
    """Izveido visu kategorijas rezultātu lapu URL sarakstu bez papildu HTTP pieprasījuma."""
    last_page = get_last_page_number(first_soup)

    if last_page == 1:
        return [start_url]

    parsed = urlsplit(start_url)
    base_path = re.sub(r"/page\d+\.html$", "", parsed.path.rstrip("/"))
    base_path = base_path.rstrip("/") + "/"

    urls = [start_url]

    for page_number in range(2, last_page + 1):
        page_path = f"{base_path}page{page_number}.html"
        page_url = urlunsplit(
            (
                parsed.scheme,
                parsed.netloc,
                page_path,
                parsed.query,
                "",
            )
        )
        urls.append(page_url)

    return urls

## 9. Funkcija `process_all_pages(start_url, delay=REQUEST_DELAY)`

Šī ir galvenā vairāku lapu rasmošanas funkcija.

Tieši šeit realizējam principu **viens URL = viens HTTP pieprasījums**:

1. `fetch_soup(start_url)` ielādē pirmo lapu **vienu reizi**;
2. `get_all_page_urls(start_url, first_soup)` no šī paša HTML nosaka visu URL sarakstu;
3. `process_page(first_soup)` no tā paša HTML iegūst pirmās lapas sludinājumus;
4. cikls sākas tikai ar `page_urls[1:]`, tātad ar otro lapu;
5. pirms katra nākamā HTTP pieprasījuma tiek veikta īsa pauze;
6. katras lapas `DataFrame` tiek saglabāts sarakstā;
7. beigās `pd.concat()` tos apvieno vienā tabulā.

Pēc apvienošanas izmetam precīzus dublikātus pēc `url`. Tā ir strukturāla drošības pārbaude, nevis analītiska datu tīrīšana.

In [ ]:
def process_all_pages(
    start_url: str,
    delay: float = REQUEST_DELAY,
) -> pd.DataFrame:
    """Lejupielādē katru rezultātu lapu vienu reizi un apvieno sludinājumus."""
    first_soup = fetch_soup(start_url)
    page_urls = get_all_page_urls(start_url, first_soup)

    print(f"Atrastas {len(page_urls)} rezultātu lapas.")

    first_df = process_page(first_soup)
    dataframes = [first_df]
    print(f"1/{len(page_urls)}: {len(first_df)} sludinājumi")

    for page_number, url in enumerate(page_urls[1:], start=2):
        sleep(delay)
        soup = fetch_soup(url)
        page_df = process_page(soup)
        dataframes.append(page_df)
        print(f"{page_number}/{len(page_urls)}: {len(page_df)} sludinājumi")

    result = pd.concat(dataframes, ignore_index=True)

    if "url" in result.columns:
        result = result.drop_duplicates(subset="url").reset_index(drop=True)

    return result

## 10. Funkcija `parse_url_metadata(start_url)`

Rezultātu faila nosaukumu nevajag studentam ievadīt ar roku, jo svarīga informācija jau atrodas SS.com URL.

Piemēram `/en/real-estate/flats/riga/centre/sell/` satur valodu `en`, galveno sadaļu `real-estate`, kategoriju `flats`, kontekstu `riga`, `centre` un darījuma tipu `sell`.

Funkcija šīs daļas sadala un atgriež vienā vārdnīcā. Zināmais SS.com termins `hand_over` failu nosaukumos tiek pārvērsts saprotamākā `rent`.

Tā pati pieeja der arī citām kategorijām, piemēram `/en/transport/cars/bmw/730/sell/`, kur kategorija būs `cars`, bet konteksts — `bmw`, `730`.

In [ ]:
def parse_url_metadata(start_url: str) -> dict:
    """No SS.com URL iegūst valodu, kategoriju, kontekstu un darījuma tipu."""
    parsed = urlsplit(start_url)

    parts = [
        unquote(part).strip().lower()
        for part in parsed.path.split("/")
        if part.strip()
    ]

    if parts and re.fullmatch(r"page\d+\.html", parts[-1]):
        parts.pop()

    language = "unknown"
    if parts and parts[0] in {"en", "lv", "ru"}:
        language = parts.pop(0)

    if len(parts) < 3:
        raise ValueError(
            "START_URL ceļā nav pietiekami daudz daļu, lai noteiktu kategoriju un darījuma tipu."
        )

    section = parts[0]
    category = parts[1]
    transaction_raw = parts[-1]
    context = parts[2:-1]

    transaction_aliases = {
        "hand_over": "rent",
        "sell": "sell",
        "buy": "buy",
        "rent": "rent",
        "exchange": "exchange",
    }

    transaction = transaction_aliases.get(transaction_raw, transaction_raw)

    safe_context = [
        re.sub(r"[^a-z0-9_-]+", "_", part).strip("_")
        for part in context
    ]

    return {
        "language": language,
        "section": section,
        "category": category,
        "context": [part for part in safe_context if part],
        "transaction": transaction,
        "transaction_raw": transaction_raw,
    }

## 11. Funkcija `build_output_paths(start_url, output_dir=OUTPUT_DIR)`

Tagad no URL metadatiem varam automātiski izveidot jēgpilnus failu nosaukumus.

Rīgas centra pārdodamo dzīvokļu piemēram rezultāts būs apmēram:

```text
data/ss_flats_riga_centre_sell_20260916_120501.csv
data/ss_flats_riga_centre_sell_20260916_120501.xlsx
```

Datums un laiks ir svarīgs, jo vairāki rasmošanas palaidieni nepārraksta iepriekšējos failus un pēc faila nosaukuma var redzēt, kad datu momentuzņēmums iegūts.

Funkcija pati failus vēl nesaglabā. Tā tikai **aprēķina divus ceļus** un atgriež tos vienā vārdnīcā.

In [ ]:
def build_output_paths(
    start_url: str,
    output_dir: Path = OUTPUT_DIR,
) -> dict:
    """Izveido timestampotus CSV un XLSX ceļus no SS.com URL metadatiem."""
    metadata = parse_url_metadata(start_url)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    name_parts = [
        "ss",
        metadata["category"],
        *metadata["context"],
        metadata["transaction"],
        timestamp,
    ]
    file_stem = "_".join(part for part in name_parts if part)

    output_dir = Path(output_dir)

    return {
        "csv": output_dir / f"{file_stem}.csv",
        "xlsx": output_dir / f"{file_stem}.xlsx",
    }

## 12. Funkcija `save_results(df, start_url, output_dir=OUTPUT_DIR)`

`save_results()` ir otrā I/O funkcija šajā notebook: tā raksta rezultātus failu sistēmā.

Funkcija ar `build_output_paths()` iegūst automātiskos failu nosaukumus, izveido rezultātu mapi, ja tā vēl nepastāv, saglabā `DataFrame` gan CSV, gan XLSX formātā un atgriež vārdnīcu ar abiem failu ceļiem.

CSV izmantojam `utf-8-sig` kodējumu. Tas saglabā Unicode tekstu un parasti ir ērti atverams arī Windows Excel vidē.

Šajā posmā mēs apzināti saglabājam **neiztīrītus analītiskos laukus**. Piemēram, cenas un platības vēl var būt teksta formā. Tieši ar šo datu kopu sāksies nākamā lekcija.

In [ ]:
def save_results(
    df: pd.DataFrame,
    start_url: str,
    output_dir: Path = OUTPUT_DIR,
) -> dict:
    """Saglabā DataFrame CSV un XLSX formātā un atgriež failu ceļus."""
    paths = build_output_paths(start_url, output_dir)
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    df.to_csv(paths["csv"], index=False, encoding="utf-8-sig")
    df.to_excel(paths["xlsx"], index=False, engine="openpyxl")

    return paths

## 13. Funkcija `run_scraping_workflow(start_url)`

Šī ir augstākā līmeņa jeb **orkestrācijas funkcija**. Tā nesatur HTML selektoru detaļas un pati neanalizē atsevišķas tabulas rindas.

Tās uzdevums ir pateikt, **kādā secībā jānotiek visam procesam**:

1. palaist `process_all_pages()`;
2. saņemt gatavu kopējo `DataFrame`;
3. palaist `save_results()`;
4. izdrukāt īsu kopsavilkumu;
5. atgriezt gala `DataFrame`.

Šāds dizains ir tipisks automatizācijas programmām: zemāka līmeņa funkcijas dara vienu konkrētu darbu, bet augstākā līmeņa funkcija tās savieno pilnā biznesa procesā.

Funkcijai ir viens obligāts arguments — sākuma URL — un viens galvenais rezultāts — `DataFrame`.

In [ ]:
def run_scraping_workflow(start_url: str) -> pd.DataFrame:
    """Izpilda pilnu SS.com rasmošanas, apvienošanas un saglabāšanas plūsmu."""
    started_at = datetime.now()

    print("Sākam SS.com datu ieguvi:")
    print(start_url)
    print()

    df = process_all_pages(start_url)
    paths = save_results(df, start_url)

    elapsed = (datetime.now() - started_at).total_seconds()

    print()
    print(f"Savākti {len(df):,} unikāli sludinājumi.")
    print(f"CSV:   {paths['csv']}")
    print(f"XLSX:  {paths['xlsx']}")
    print(f"Izpildes ilgums: {elapsed:.1f} s")

    return df

## Pilnā plūsma

Šī ir vienīgā šūna, kas normālā **Run All** izpildē sāk HTTP pieprasījumus.

Ja vēlamies rasmošanas mērķi mainīt, mainām `START_URL` konfigurācijas šūnā un palaižam notebook no sākuma.

Pirmā lapa tiek lejupielādēta tieši vienu reizi. Tās `BeautifulSoup` objekts tiek izmantots gan lapošanas informācijai, gan pirmās lapas sludinājumu apstrādei. Pārējās lapas cikls sāk apstrādāt tikai no otrā URL.

Pēc izpildes `df` paliek atmiņā un ir gatavs nākamās lekcijas darbam ar:

- datu tipiem;
- trūkstošām vērtībām;
- cenu un platību tīrīšanu;
- grupēšanu un agregāciju;
- `matplotlib` vizualizācijām.

In [ ]:
# ============================================================
# FULL SCRAPING WORKFLOW
# ============================================================

df = run_scraping_workflow(START_URL)

print()
print("DataFrame izmērs:", df.shape)
display(df.head())